## 构建检索增强生成 (RAG) 应用程序

将适当的信息引入并将其插入模型提示的过程称为检索增强生成 (RAG)。

In [1]:
# import os

# os.environ["HTTP_PROXY"] = ''
# os.environ["HTTPS_PROXY"] = ''
# os.environ["all_proxy"] = ''
# os.environ["ALL_PROXY"] = ''

In [1]:
import os
from langchain_ollama import ChatOllama

from langchain_ollama import OllamaEmbeddings


model = ChatOllama(
    model="llama3.1",
    # model="qwen2.5:14b",
    # temperature=0,
    # other params...
)

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = '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'

In [3]:
import bs4
from langchain import hub
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders.word_document import Docx2txtLoader, UnstructuredWordDocumentLoader

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load, chunk and index the contents of the blog.
# loader = WebBaseLoader(
#     # web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
#     web_paths=('https://new.qq.com/rain/a/20240325A07L7L00', ),
#     # bs_kwargs=dict(
#     #     parse_only=bs4.SoupStrainer(
#     #         class_=("Post-RichTextContainer", "Post-Title", "Post-Header")
#     #     )
#     # ),
# )

# loader = UnstructuredWordDocumentLoader("./data_test/fake.docx")

# docs = loader.load()
# # print(docs)
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
# splits = text_splitter.split_documents(docs)
# vectorstore = Chroma.from_documents(documents=splits, embedding=OllamaEmbeddings(model="llama3.1",))

# # Retrieve and generate using the relevant snippets of the blog.
# retriever = vectorstore.as_retriever()
# prompt = hub.pull("rlm/rag-prompt")


# def format_docs(docs):
#     return "\n\n".join(doc.page_content for doc in docs)


# rag_chain = (
#     {"context": retriever | format_docs, "question": RunnablePassthrough()}
#     | prompt
#     | model
#     | StrOutputParser()
# )

# # rag_chain.invoke("What is Task Decomposition?")
# rag_chain.invoke("2024年中国医疗器械产业市场规模")

# 详细演练

# 1. 索引：加载

In [4]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders.word_document import Docx2txtLoader, UnstructuredWordDocumentLoader

# Only keep post title, headers, and content from the full HTML.
# bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
# loader = WebBaseLoader(
#     web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
#     bs_kwargs={"parse_only": bs4_strainer},
# )
# docs = loader.load()

# len(docs[0].page_content)


# load docx file test
# loader = UnstructuredWordDocumentLoader('./data_test/2024年医疗器械强制性行业标准制修订计划项目.docx')
# docs = loader.load()
# print(docs[0].page_content[:500])

## 调用api解析文档

In [4]:
import unstructured_client
from unstructured_client.models import operations, shared

client = unstructured_client.UnstructuredClient(
    api_key_auth="qGhRbvPkcUk121T51CRab1bNaNrLkP",
    server_url="https://api.unstructuredapp.io",
)

filename = "./data_test/医疗器械经营质量管理规范.docx"
with open(filename, "rb") as f:
    data = f.read()

req = operations.PartitionRequest(
    partition_parameters=shared.PartitionParameters(
        files=shared.Files(
            content=data,
            file_name=filename,
        ),
        # --- Other partition parameters ---
        # Note: Defining 'strategy', 'chunking_strategy', and 'output_format'
        # parameters as strings is accepted, but will not pass strict type checking. It is
        # advised to use the defined enum classes as shown below.
        strategy=shared.Strategy.HI_RES,  
        # languages=['zh'],
    ),
)

# try:
#     res = client.general.partition(request=req)
#     print(res.elements[0])
# except Exception as e:
#     print(e)

# 本地加载文档

In [5]:
import nltk
# nltk.download()

# # nltk.set_proxy('127.0.0.1:7890')
# nltk.set_proxy('127.0.0.1:58591')
# 下载对于的包
# nltk.download('punkt/PY3_tab')
# # nltk.find('./nltk_data')

In [7]:
from langchain_community.document_loaders import UnstructuredWordDocumentLoader

filename = "./data_test/医疗器械经营质量管理规范.docx"
# loader = UnstructuredWordDocumentLoader(filename, mode="elements")
loader = UnstructuredWordDocumentLoader(filename)
docs = loader.load()

In [5]:
# 
# from langchain_community.document_loaders import Docx2txtLoader
# filename = "./data_test/2024年医疗器械强制性行业标准制修订计划项目.docx"
# loader = Docx2txtLoader(filename)

# data = loader.load()

# data

## 2. 索引：拆分
我们加载的文档超过 42k 个字符。这对于许多模型的上下文窗口来说太长了。即使对于那些能够在其上下文窗口中容纳完整文章的模型，模型也可能难以在非常长的输入中找到信息。

为了解决这个问题，我们将文档拆分成块以进行嵌入和向量存储。这应该有助于我们在运行时仅检索博客文章中最相关的部分。


在本例中，我们将文档拆分成 1000 个字符的块，块之间有 200 个字符的重叠。重叠有助于减轻将语句与其相关的重要上下文分开的可能性。我们使用RecursiveCharacterTextSplitter，它将使用常见的分隔符（如换行符）递归地拆分文档，直到每个块的大小都合适。这是推荐用于通用文本用例的文本拆分器。

我们设置add_start_index=True，以便每个拆分文档在初始文档中开始的字符索引作为元数据属性“start_index”保留下来。

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    # chunk_size=1000, chunk_overlap=200, add_start_index=True
    chunk_size=500, chunk_overlap=100, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

len(all_splits)

40

In [9]:
all_splits[0]

Document(metadata={'source': './data_test/医疗器械经营质量管理规范.docx', 'start_index': 0}, page_content='附件\n\n医疗器械经营质量管理规范\n\n第一章  总  则\n\n为了加强医疗器械经营质量管理，规范医疗器械经营活动，保证医疗器械安全、有效，根据《医疗器械监督管理条例》《医疗器械经营监督管理办法》等法规规章的规定，制定本规范。\n\n本规范是医疗器械经营质量管理的基本要求。从事医疗器械经营活动，应当在医疗器械采购、验收、贮存、销售、运输、售后服务等全过程采取有效的质量管理措施，确保医疗器械产品在经营过程中的质量安全与可追溯。\n\n医疗器械经营企业应当严格执行本规范。\n\n医疗器械注册人、备案人销售其注册或者备案的医疗器械，以及医疗器械流通过程中其他涉及贮存与运输医疗器械的，应当符合本规范的相关要求。\n\n医疗器械注册人、备案人依法对上市医疗器械的安全、有效负责，医疗器械经营企业（以下简称企业）对本企业的经营行为负责。\n\n从事医疗器械经营活动，应当按照所经营医疗器械的风险程度实行风险管理，并采取相应的质量管理措施。\n\n\n\n企业及其从业者应当诚实守信、依法经营，禁止任何虚假、欺骗行为。\n\n鼓励企业使用信息化手段传递和存储相关政府管理部门制作的电子证照。\n\n电子证照与纸质证书具有同等法律效力。')

In [10]:
len(all_splits[0].page_content)

493

In [11]:
all_splits[10].metadata

{'source': './data_test/医疗器械经营质量管理规范.docx', 'start_index': 3890}

## 3. 索引：存储

现在我们需要索引我们的 66 个文本块，以便我们可以在运行时对其进行搜索。最常见的方法是嵌入每个文档拆分的内容并将这些嵌入插入向量数据库（或向量存储）。当我们想要搜索我们的拆分时，我们会获取一个文本搜索查询，将其嵌入，并执行某种“相似性”搜索以识别与我们的查询嵌入最相似的存储拆分。最简单的相似性度量是余弦相似度 - 我们测量每对嵌入（它们是高维向量）之间的角度的余弦。

我们可以使用 Chroma 向量数据库和 OpenAIEmbeddings 模型在一个命令中嵌入和存储所有文档拆分。

In [12]:
from langchain_chroma import Chroma
# from langchain_openai import OpenAIEmbeddings
from langchain_ollama import OllamaEmbeddings

vectorstore = Chroma.from_documents(documents=all_splits, embedding=OllamaEmbeddings(model="llama3.1"))

------------------------
http://127.0.0.1:11434/api/embed
------------------------


## 4. 检索和生成：检索

In [32]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})

retrieved_docs = retriever.invoke("医疗器械经营质量管理规范是什么？")

len(retrieved_docs)

6

In [33]:
print(retrieved_docs[0].page_content)

第一百一十一条　企业应当协助医疗器械注册人、备案人履行召回义务，按照召回计划的要求及时传达、反馈医疗器械召回信息，控制和收回存在质量安全隐患的医疗器械，并建立医疗器械召回记录。

第十章  附  则

第一百一十二条　本规范下列用语的含义是：

（一）在职：与企业确定劳动关系的在册人员；



（二）在岗：相关岗位人员在工作时间内在规定的岗位履行职责。

第一百一十三条　从事医疗器械网络销售的，除应当符合本规范相关要求外，还应当遵守相关法律、法规、规章、规范的有关规定。

第一百一十四条　为医疗器械注册人、备案人和经营企业专门提供医疗器械运输、贮存服务的企业，应当遵守本规范及相应附录的要求。

为使用单位专门提供医疗器械运输、贮存服务的企业，参照执行本规范及相应附录的要求。

第一百一十五条　省级药品监督管理部门可以根据本规范制定适用本辖区的医疗器械经营质量管理相关规定。

第一百一十六条　本规范自2024年7月1日起施行。2014年12月12日原国家食品药品监督管理总局《关于施行医疗器械经营质量管理规范的公告》（2014年第58号）同时废止。


## 5. 检索和生成：生成

In [11]:
from langchain import hub

# 
prompt = hub.pull("rlm/rag-prompt")

example_messages = prompt.invoke(
    {"context": "filler context", "question": "filler question"}
).to_messages()

example_messages

/home/zj/python-venvs/pyRobBot/lib/python3.10/site-packages/langsmith/client.py:5301: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  prompt = loads(json.dumps(prompt_object.manifest))


[HumanMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: filler question \nContext: filler context \nAnswer:", additional_kwargs={}, response_metadata={})]

In [12]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

In [13]:
print(example_messages[0].content)

You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: filler question 
Context: filler context 
Answer:


In [38]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

for chunk in rag_chain.stream("医疗器械经营质量管理规范是什么？"):
    print(chunk, end="", flush=True)

It seems you provided a large text block related to "医疗器械经营质量管理规范" (Medical Device Sales Quality Management Standard) in China, but didn't ask a specific question. I'll provide an answer based on the content of the text.

The provided text is a standard for medical device sales quality management in China, outlining the requirements and responsibilities for medical device sales companies. It covers topics such as:

1. Quality management personnel qualifications
2. Quality management system establishment and improvement
3. Risk management and quality control measures
4. Record-keeping and archiving of quality-related information

Based on this content, here's an answer to a hypothetical question:

**What are the requirements for medical device sales companies in China regarding quality management?**

According to Article 2 of the standard, medical device sales companies must establish a quality management system that meets the requirements outlined in the standard. This includes ensurin

In [44]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "您是问答任务的助手。 使用以下检索到的上下文来回答问题。 如果您不知道答案，请说您不知道。 最多使用三句话并保持答案简洁。\n\n {context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)


question_answer_chain = create_stuff_documents_chain(model, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

response = rag_chain.invoke({"input": "医疗器械经营质量管理规范有几条要求"})
print(response["answer"])



回答：23条


In [1]:
for document in response["context"]:
    print(document)
    print('-' * 100)

NameError: name 'response' is not defined